In [76]:
import xarray as xr
import numpy as np

In [77]:
files = []

filenames = [4,5,6,7,8,9,10,11,14,15]

filenames = [f"./tests/files_for_v1_paper/20.0sens1022_{filename}_Regional.nc/" for filename in filenames]

default_file = xr.open_dataset("./tests/files_for_v1_paper/20.0sens1022_1_Regional.nc",engine="netcdf4")

obs_file = xr.open_dataset("./tests/files_for_v1_paper/20.0_OBS.nc",engine="netcdf4")

temp_encoding = default_file["SWCF_1_1"].encoding
encoding_dict = {var: temp_encoding for var in default_file.data_vars}
for filename in filenames:
    files.append(xr.open_dataset(filename,engine="netcdf4"))

/home/nobrewi2/.pyenv/versions/3.12.0/envs/env_quadtune/lib/python3.12/site-packages/xarray/conventions.py:204: SerializationWarning: variable 'PRECT_GLB' has multiple fill values {np.float32(1e+20), np.float64(1e+20)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)
/home/nobrewi2/.pyenv/versions/3.12.0/envs/env_quadtune/lib/python3.12/site-packages/xarray/conventions.py:204: SerializationWarning: variable 'PRECT_DYCOMS' has multiple fill values {np.float32(1e+20), np.float64(1e+20)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)
/home/nobrewi2/.pyenv/versions/3.12.0/envs/env_quadtune/lib/python3.12/site-packages/xarray/conventions.py:204: SerializationWarning: variable 'PRECT_HAWAII' has multiple fill values {np.float32(1e+20), np.float64(1e+20)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)
/home/nobrewi2/.pyenv/versions/3.12.0/envs/env_quadtune/lib/python3.12/site-packages/xarray/conventions.py:204: Ser

In [78]:

obs_file.weights.values

array(0.7853882)

In [79]:
metric_names = [f"SWCF_{i}_{j}" for i in range(1,10) for j in range(1,19)]

obs_weights_names = [f"weights_{i}_{j}_SWCF" for i in range(1,10) for j in range(1,19)]

numb_names = [f"numb_{i}_{j}" for i in range(1,10) for j in range(1,19)]

In [80]:
default_file

<xarray.Dataset> Size: 9kB
Dimensions:                           (col: 1)
Dimensions without coordinates: col
Data variables: (12/2325)
    SWCF_GLB                          (col) float32 4B ...
    SWCF_DYCOMS                       (col) float32 4B ...
    numb_DYCOMS                       (col) float32 4B ...
    SWCF_HAWAII                       (col) float32 4B ...
    numb_HAWAII                       (col) float32 4B ...
    SWCF_VOCAL                        (col) float32 4B ...
    ...                                ...
    clubb_c_wp2_splat                 (col) float32 4B ...
    cldfrc_dp2                        (col) float32 4B ...
    clubb_c_invrs_tau_n2              (col) float32 4B ...
    clubb_z_displace                  (col) float32 4B ...
    micro_mg_dcs                      (col) float32 4B ...
    micro_mg_autocon_lwp_exp          (col) float32 4B ...

In [81]:


parameter_names = sorted([v for v in files[0].data_vars if v == v.lower() and v.split("_")[0] != 'numb'])
default_params = default_file[parameter_names].to_array().values.flatten()

parameter_names

['cldfrc_dp2',
 'clubb_altitude_threshold',
 'clubb_c11',
 'clubb_c7',
 'clubb_c8',
 'clubb_c_invrs_tau_bkgnd',
 'clubb_c_invrs_tau_n2',
 'clubb_c_invrs_tau_n2_clear_wp3',
 'clubb_c_invrs_tau_n2_wp2',
 'clubb_c_invrs_tau_n2_wpxp',
 'clubb_c_invrs_tau_n2_xp2',
 'clubb_c_invrs_tau_sfc',
 'clubb_c_invrs_tau_shear',
 'clubb_c_invrs_tau_wpxp_n2_thresh',
 'clubb_c_k10',
 'clubb_c_k8',
 'clubb_c_wp2_splat',
 'clubb_gamma_coef',
 'clubb_nu1',
 'clubb_nu2',
 'clubb_z_displace',
 'micro_mg_autocon_lwp_exp',
 'micro_mg_dcs']

In [82]:
param_vals = []

for file in files:
    param_vals.append(file[parameter_names].to_array().values.flatten())



param_vals.append(default_params)
# param_vals = np.array(param_vals).reshape((len(files),len(parameter_names)))
param_vals = np.array(param_vals)
param_vals.dtype

dtype('float32')

In [83]:
metric_vals = []

for file in files:
    metric_vals.append(file[metric_names].to_array().values)

metric_vals = np.array(metric_vals).reshape((len(files),len(metric_names)))
metric_vals

array([[-43.40673  , -40.804348 , -38.189987 , ..., -12.159522 ,
        -12.957233 ,  -3.8934574],
       [-45.522976 , -42.07738  , -37.652042 , ...,  -9.900849 ,
        -10.974037 ,  -3.6806881],
       [-43.765686 , -40.729153 , -38.52785  , ...,  -9.639808 ,
        -11.654156 ,  -3.6966207],
       ...,
       [-45.029324 , -40.178913 , -35.32407  , ...,  -9.098693 ,
        -10.9540825,  -3.418376 ],
       [-46.537952 , -43.026424 , -39.4985   , ..., -12.517888 ,
        -14.025725 ,  -4.334824 ],
       [-43.363277 , -40.47431  , -37.129276 , ..., -10.897968 ,
        -12.832737 ,  -3.7648532]], shape=(10, 162), dtype=float32)

In [84]:
obs_metrics = obs_file[metric_names].to_array()[:,0]

obs_weights = obs_file[obs_weights_names].to_array()

In [85]:
param_vals.dtype

dtype('float32')

In [86]:
timesize = 5
productsize = 3
enssize = len(files)
inputparamsize = len(parameter_names)

param_vars = {}
data_vars = {}

for i , var_name in enumerate(metric_names):
    empty_data = np.full((timesize,productsize,enssize+1),np.nan)

    empty_data[0,0,:-1] = metric_vals[:,i]

    empty_data[0,0,-1] = default_file[var_name]

    data_vars[var_name] = (("time","product","ens_idx"),empty_data)




ds = xr.Dataset(
    data_vars = data_vars,
    coords={
        'time':['ANN','DJF',"JJA","MAM","SON"],
        "product":["mod","obs","sur"],
        "ens_idx":[*[str(t) for t in np.arange(enssize)],'ctrl'],
    }
)


ds_params = xr.Dataset(
    coords={
        'time':['ANN','DJF',"JJA","MAM","SON"],
        "product":["mod","obs","sur"],
        "ens_idx":[*[str(t) for t in np.arange(enssize)],'ctrl'],
    }
)


for name, value in zip(metric_names, obs_metrics):
    ds[name][0,1,:] = value



for name, value in zip(numb_names,default_file[numb_names].to_array()):
    ds[name] = value


In [87]:
ds_params["params"] = (("ens_idx","input_params"), param_vals)




obsWeights = obs_file[obs_weights_names].to_array()



ds["obs_weights"] = obsWeights

In [88]:
obsWeights

<xarray.DataArray (variable: 162)> Size: 1kB
array([0.17281051, 0.17281051, 0.17281051, 0.17281051, 0.17281051,
       0.17281051, 0.17281051, 0.17281051, 0.17281051, 0.17281051,
       0.17281051, 0.17281051, 0.17281051, 0.17281051, 0.17281051,
       0.17281051, 0.17281051, 0.17281051, 0.49749846, 0.49749846,
       0.49749846, 0.49749846, 0.49749846, 0.49749846, 0.49749846,
       0.49749846, 0.49749846, 0.49749846, 0.49749846, 0.49749846,
       0.49749846, 0.49749846, 0.49749846, 0.49749846, 0.49749846,
       0.49749846, 0.76218426, 0.76218426, 0.76218426, 0.76218426,
       0.76218426, 0.76218426, 0.76218426, 0.76218426, 0.76218426,
       0.76218426, 0.76218426, 0.76218426, 0.76218426, 0.76218426,
       0.76218426, 0.76218426, 0.76218426, 0.76218426, 0.93494475,
       0.93494475, 0.93494475, 0.93494475, 0.93494475, 0.93494475,
       0.93494475, 0.93494475, 0.93494475, 0.93494475, 0.93494475,
       0.93494475, 0.93494475, 0.93494475, 0.93494475, 0.93494475,
       0.93494475, 0.93494475, 0.9949437 , 0.9949437 , 0.9949437 ,
       0.9949437 , 0.9949437 , 0.9949437 , 0.9949437 , 0.9949437 ,
       0.9949437 , 0.9949437 , 0.9949437 , 0.9949437 , 0.9949437 ,
       0.9949437 , 0.9949437 , 0.9949437 , 0.9949437 , 0.9949437 ,
       0.93494475, 0.93494475, 0.93494475, 0.93494475, 0.93494475,
       0.93494475, 0.93494475, 0.93494475, 0.93494475, 0.93494475,
       0.93494475, 0.93494475, 0.93494475, 0.93494475, 0.93494475,
       0.93494475, 0.93494475, 0.93494475, 0.76218426, 0.76218426,
       0.76218426, 0.76218426, 0.76218426, 0.76218426, 0.76218426,
       0.76218426, 0.76218426, 0.76218426, 0.76218426, 0.76218426,
       0.76218426, 0.76218426, 0.76218426, 0.76218426, 0.76218426,
       0.76218426, 0.49749846, 0.49749846, 0.49749846, 0.49749846,
       0.49749846, 0.49749846, 0.49749846, 0.49749846, 0.49749846,
       0.49749846, 0.49749846, 0.49749846, 0.49749846, 0.49749846,
       0.49749846, 0.49749846, 0.49749846, 0.49749846, 0.17281051,
       0.17281051, 0.17281051, 0.17281051, 0.17281051, 0.17281051,
       0.17281051, 0.17281051, 0.17281051, 0.17281051, 0.17281051,
       0.17281051, 0.17281051, 0.17281051, 0.17281051, 0.17281051,
       0.17281051, 0.17281051])
Coordinates:
  * variable  (variable) object 1kB 'weights_1_1_SWCF' ... 'weights_9_18_SWCF'
    height    float64 8B ...
Attributes: (12/57)
    CDI:                        Climate Data Interface version 1.9.6 (http://...
    Conventions:                CF-1.4
    history:                    Mon Jul 28 13:37:48 2025: ncks -A local_2_9_1...
    institution:                NASA Langley Research Center
    title:                      CERES EBAF TOA and Surface Fluxes. Monthly Av...
    comment:                    Climatology from 07/2005 to 06/2015
    ...                         ...
    Author:                     Gregory CESANA, Helene CHEPFER, LMD/IPSL
    Scientific_contact:         helene.chepfer@lmd.polytechnique.fr
    Technical_support:          gregory.cesana@lmd.polytechnique.fr
    Creationdate:               20110323
    Website:                    http://climserv.ipsl.polytechnique.fr/fr/cfmi...
    nco_openmp_thread_number:   1

In [93]:
# encoding = {var:{'dtype':'float32','_FillValue':-9999.0} for var in ds_params.data_vars}
# ds_params.to_netcdf("ppe_like_params.nc",encoding=encoding)
ds_params.to_netcdf("ppe_like_params.nc")

# encoding = {var:{'dtype':'float32','_FillValue':-9999.0} for var in ds.data_vars}

# ds.to_netcdf("ppe_like_metrics.nc",encoding=encoding)
ds.to_netcdf("ppe_like_metrics.nc")

In [91]:
obs_file[obs_weights_names]

<xarray.Dataset> Size: 1kB
Dimensions:            ()
Coordinates:
    height             float64 8B ...
Data variables: (12/162)
    weights_1_1_SWCF   float64 8B 0.1728
    weights_1_2_SWCF   float64 8B 0.1728
    weights_1_3_SWCF   float64 8B 0.1728
    weights_1_4_SWCF   float64 8B 0.1728
    weights_1_5_SWCF   float64 8B 0.1728
    weights_1_6_SWCF   float64 8B 0.1728
    ...                 ...
    weights_9_13_SWCF  float64 8B 0.1728
    weights_9_14_SWCF  float64 8B 0.1728
    weights_9_15_SWCF  float64 8B 0.1728
    weights_9_16_SWCF  float64 8B 0.1728
    weights_9_17_SWCF  float64 8B 0.1728
    weights_9_18_SWCF  float64 8B 0.1728
Attributes: (12/57)
    CDI:                        Climate Data Interface version 1.9.6 (http://...
    Conventions:                CF-1.4
    history:                    Mon Jul 28 13:37:48 2025: ncks -A local_2_9_1...
    institution:                NASA Langley Research Center
    title:                      CERES EBAF TOA and Surface Fluxes. Monthly Av...
    comment:                    Climatology from 07/2005 to 06/2015
    ...                         ...
    Author:                     Gregory CESANA, Helene CHEPFER, LMD/IPSL
    Scientific_contact:         helene.chepfer@lmd.polytechnique.fr
    Technical_support:          gregory.cesana@lmd.polytechnique.fr
    Creationdate:               20110323
    Website:                    http://climserv.ipsl.polytechnique.fr/fr/cfmi...
    nco_openmp_thread_number:   1

In [92]:
param_vals.shape

(11, 23)